# Milestone 5 — Tools ausbauen + Memory

## Setup: Agent-Grundlage aus M3 wiederherstellen


In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(dotenv_path="../.env")
client = OpenAI()

chroma_client = chromadb.PersistentClient(path="../data/chroma_db")
collection = chroma_client.get_or_create_collection(name="health_fitness_videos")


def embed_query(text: str) -> list:
    response = client.embeddings.create(model="text-embedding-3-small", input=[text])
    return response.data[0].embedding


def search_video(query: str, n_results: int = 3) -> list:
    query_embedding = embed_query(query)
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    matches = []
    for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
        matches.append({"text": doc, "start": metadata["start"], "end": metadata["end"]})
    return matches


@tool
def search_video_tool(query: str) -> str:
    """Durchsucht das Transcript des Videos nach Informationen zu einer bestimmten Frage oder einem Thema.
    Gib eine natürlichsprachliche Frage oder ein Stichwort ein. Gibt relevante Textausschnitte mit Zeitstempeln zurück."""
    results = search_video(query, n_results=3)
    formatted = ""
    for r in results:
        formatted += f"[{r['start']:.1f}s - {r['end']:.1f}s]: {r['text']}\n\n"
    return formatted


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [search_video_tool]
checkpointer = MemorySaver()
agent = create_agent(llm, tools, checkpointer=checkpointer)

print("✅ Vollständige Grundlage geladen:", collection.count(), "Chunks verfügbar")

✅ Vollständige Grundlage geladen: 194 Chunks verfügbar


## Memory hinzufügen

Wir nutzen LangGraphs `MemorySaver` (In-Memory-Checkpointer) und geben dem Agent bei jedem Aufruf eine `thread_id` mit, damit jedes Gespräch getrennt bleibt.


In [3]:
from langgraph.checkpoint.memory import MemorySaver

# Der Checkpointer speichert den Gesprächsverlauf pro thread_id im Arbeitsspeicher
checkpointer = MemorySaver()

# Agent neu erstellen, diesmal MIT Checkpointer
agent = create_agent(llm, tools, checkpointer=checkpointer)


def ask_agent(question: str, thread_id: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    return response["messages"][-1].content


# Test: zwei Fragen im SELBEN Thread -- die zweite bezieht sich auf die erste
my_thread = "test-conversation-1"

answer1 = ask_agent("What foods are good for brain health?", thread_id=my_thread)
print("Antwort 1:", answer1[:150], "...\n")

answer2 = ask_agent("How much of it should I take daily?", thread_id=my_thread)
print("Antwort 2:", answer2)

Antwort 1: Foods that are good for brain health include:

1. **Berries**: Particularly blueberries and other dark berries, which are often highlighted for their  ...

Antwort 2: The excerpts did not provide specific daily intake recommendations for the foods beneficial for brain health. However, it was mentioned that for certain supplements, like creatine, a threshold of five grams per day is suggested. 

For general guidance on foods like berries and other brain-healthy options, it's advisable to incorporate them into your diet regularly, focusing on moderation and variety. A balanced approach that includes a range of these foods, while also considering personal preferences and dietary needs, is key to supporting brain health.


## Multi-Query Retrieval

Statt nur mit der Original-Frage zu suchen, generieren wir mehrere Umformulierungen und kombinieren die Ergebnisse.


In [7]:
def generate_query_variations(question: str, n_variations: int = 3) -> list:
    prompt = f"""Generate {n_variations} different ways to search for information related to this question.
Each variation should use different wording but search for the same underlying information.
Return ONLY the variations, one per line, no numbering, no extra text.

Question: {question}"""

    response = llm.invoke(prompt)

    # Antwort in einzelne Zeilen aufteilen, leere Zeilen rausfiltern
    variations = [line.strip() for line in response.content.split("\n") if line.strip()]

    # Die Original-Frage nehmen wir immer mit dazu, nicht nur die Umformulierungen
    return [question] + variations


def multi_query_search(question: str, n_results_per_query: int = 3) -> list:
    queries = generate_query_variations(question)

    all_results = []
    seen_texts = set()  # verhindert doppelte Chunks in der finalen Liste

    for query in queries:
        results = search_video(query, n_results=n_results_per_query)
        for r in results:
            if r["text"] not in seen_texts:
                all_results.append(r)
                seen_texts.add(r["text"])

    return all_results


# Test: Vergleich normale Suche vs. Multi-Query
question = "How much creatine should I take?"

normal_results = search_video(question)
multi_results = multi_query_search(question)

print(f"Normale Suche: {len(normal_results)} Treffer")
print(f"Multi-Query Suche: {len(multi_results)} Treffer (dedupliziert)\n")

for r in multi_results:
    print(f"[{r['start']:.1f}s] {r['text'][:100]}...\n")

Normale Suche: 3 Treffer
Multi-Query Suche: 4 Treffer (dedupliziert)

[2124.1s] in people that aren't getting creatine from animal sources. And there's some evidence detailed withi...

[2028.6s] The first author is Roschel, R-O-S-C-H-E-L. We will provide a link to this study, rather, this revie...

[2061.2s] Now, the most typical form of creatine is so-called creatine monohydrate. There are other forms of c...

[1930.9s] But because, fortunately, at least, not yet, or not to my awareness, I'm not suffering from any cogn...



## Zeitstempel-Suche

Findet den Chunk, der zu einem bestimmten Zeitpunkt im Video gehört -- reine Metadata-Filterung, kein Embedding nötig.


In [8]:
def get_all_chunks_with_metadata() -> list:
    # .get() statt .query() -- wir wollen ALLE Chunks, keine semantische Suche
    results = collection.get(include=["documents", "metadatas"])

    chunks = []
    for doc, metadata in zip(results["documents"], results["metadatas"]):
        chunks.append({
            "text": doc,
            "start": metadata["start"],
            "end": metadata["end"],
        })
    return chunks


def search_by_timestamp(seconds: float) -> dict | None:
    all_chunks = get_all_chunks_with_metadata()

    for chunk in all_chunks:
        if chunk["start"] <= seconds <= chunk["end"]:
            return chunk

    return None  # falls der Zeitpunkt außerhalb des Videos liegt


# Test: Was wurde bei Minute 35 (= 2100 Sekunden) gesagt?
result = search_by_timestamp(2100)

if result:
    print(f"[{result['start']:.1f}s - {result['end']:.1f}s]")
    print(result["text"])
else:
    print("Kein Chunk für diesen Zeitpunkt gefunden.")

[2093.6s - 2124.1s]
there are these theories that creatine can cause hair loss. And indeed, for people that are very DHT sensitive, it might. There's going to be a lot of variation person to person in terms of how much creatine impacts DHT, and how many DHT receptors they have on their scalp, and therefore, whether or not they experience hair loss. I'm just giving you all this information, so that you're aware of the various things that creatine can do. But nonetheless, I think it's interesting that creatine supplementation of five grams per day, that's creatine monohydrate, has been shown to improve cognition


## Summary-Tool

Generiert eine strukturierte Zusammenfassung aus dem gesamten Transcript -- passt die Struktur automatisch an den Inhalt an (Rezept/Dosierung/Übung/allgemein).


In [9]:
def get_full_transcript_text() -> str:
    all_chunks = get_all_chunks_with_metadata()
    # Chunks nach Startzeit sortieren, damit die Reihenfolge stimmt
    sorted_chunks = sorted(all_chunks, key=lambda c: c["start"])
    return "\n".join(chunk["text"] for chunk in sorted_chunks)


@tool
def summarize_video_tool(focus: str = "general") -> str:
    """Erstellt eine Zusammenfassung des gesamten Videos.
    Nutze dieses Tool, wenn der User um eine Zusammenfassung, einen Überblick, oder die Kernaussagen des Videos bittet.
    'focus' kann 'general' (allgemeiner Überblick) oder 'technical' (strukturierte Extraktion konkreter Details wie Zutaten, Dosierungen, Übungen) sein."""

    full_text = get_full_transcript_text()

    if focus == "technical":
        prompt = f"""Analysiere dieses Transcript und extrahiere die konkreten, umsetzbaren Informationen in strukturierter Form.
Falls es sich um ein Rezept handelt: liste Zutaten und Schritte.
Falls es sich um Nahrungsergänzungsmittel/Dosierungen handelt: liste Substanz, empfohlene Menge, und Kontext.
Falls es sich um Trainingsübungen handelt: liste Übung, Wiederholungen, Hinweise.
Bei anderen Inhalten: extrahiere die wichtigsten konkreten Fakten/Empfehlungen in Stichpunkten.

Transcript:
{full_text}"""
    else:
        prompt = f"""Fasse dieses Video in 3-5 Sätzen zusammen. Nenne die wichtigsten besprochenen Themen.

Transcript:
{full_text}"""

    response = llm.invoke(prompt)
    return response.content


# Test: beide Varianten
general_summary = summarize_video_tool.invoke({"focus": "general"})
print("=== ALLGEMEINE ZUSAMMENFASSUNG ===")
print(general_summary)

technical_summary = summarize_video_tool.invoke({"focus": "technical"})
print("\n=== TECHNISCHE ZUSAMMENFASSUNG ===")
print(technical_summary)

=== ALLGEMEINE ZUSAMMENFASSUNG ===
In der aktuellen Episode des Huberman Lab Podcasts spricht Andrew Huberman über die Beziehung zwischen Ernährung und Gehirngesundheit. Er erläutert, welche Nahrungsmittel die kognitive Funktion und das allgemeine Wohlbefinden fördern können, und beschreibt drei Hauptfaktoren, die unsere Nahrungswahl beeinflussen: Signale aus dem Darm, die metabolische Verfügbarkeit von Lebensmitteln und die Überzeugung über deren gesundheitliche Vorteile. Huberman hebt hervor, dass essentielle Fettsäuren, insbesondere Omega-3-Fettsäuren, Phosphatidylserin und Cholin, entscheidend für die neuronale Gesundheit sind. Zudem wird diskutiert, wie man seine Vorlieben für bestimmte Nahrungsmittel ändern kann, um gesündere Entscheidungen zu treffen, indem man sie mit Lebensmitteln kombiniert, die den Blutzuckerspiegel erhöhen und somit die Gehirnmetabolismus aktivieren.

=== TECHNISCHE ZUSAMMENFASSUNG ===
Hier ist die strukturierte Analyse des Transkripts mit den extrahierten,

## Metadata-Tool — vollständig

Wir holen zusätzliche Video-Metadata über yt-dlp (Kanal, Upload-Datum, Beschreibung, Länge) und speichern sie einmalig als JSON-Datei -- kein erneuter Download nötig, nur Metadaten-Abfrage.


In [15]:
import yt_dlp
import json
final_video_id = "E7W4OQfJWdw"

In [17]:
def fetch_video_metadata(video_id: str) -> dict:
    url = f"https://www.youtube.com/watch?v={video_id}"
    ydl_opts = {"quiet": True, "skip_download": True}  # skip_download: nur Infos holen, keine Datei laden

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    return {
        "video_id": video_id,
        "title": info.get("title"),
        "channel": info.get("channel") or info.get("uploader"),
        "upload_date": info.get("upload_date"),      # Format: YYYYMMDD
        "description": info.get("description"),
        "duration_seconds": info.get("duration"),
    }


# Einmalig abrufen und speichern
video_metadata = fetch_video_metadata(final_video_id)

os.makedirs("../data/video_metadata", exist_ok=True)
with open(f"../data/video_metadata/{final_video_id}.json", "w", encoding="utf-8") as f:
    json.dump(video_metadata, f, ensure_ascii=False, indent=2)

print(video_metadata)

{'video_id': 'E7W4OQfJWdw', 'title': 'Nutrients For Brain Health & Performance | Huberman Lab Podcast #42', 'channel': 'Andrew Huberman', 'upload_date': '20211018', 'description': 'This episode I describe science-supported nutrients for brain and performance (cognition) and for nervous system health generally.\n\nI describe 10 tools for this purpose, including specific amounts and sources for Omega-3 fatty acids which make up the "structural fat" of neurons (nerve cells) and allow them to function across our lifespan. I also review data on creatine, phosphatidylserine, anthocyanins, choline, glutamine and how they each impact brain function in healthy people seeking to reinforce and improve their cognition and in those combatting cognitive decline. I describe both food-based and supplement-based sources for these compounds, and their effective dose ranges based on peer-reviewed literature.\n\nThen I review the 3 factors: gut-brain signaling, perceived taste, and learned associations th

## Metadata-Tool als Agent-Tool verpacken


In [18]:
@tool
def get_video_metadata_tool() -> str:
    """Gibt Informationen über das Video selbst zurück: Titel, Kanal, Upload-Datum, Länge, Beschreibung.
    Nutze dieses Tool, wenn der User nach dem Video selbst fragt (nicht nach seinem Inhalt) -- 
    z.B. 'Wie heißt das Video?', 'Wie lang ist es?', 'Von wem ist es?', 'Wann wurde es hochgeladen?'."""

    with open(f"../data/video_metadata/{final_video_id}.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

    duration_min = metadata["duration_seconds"] // 60
    upload_date_formatted = f"{metadata['upload_date'][:4]}-{metadata['upload_date'][4:6]}-{metadata['upload_date'][6:]}"

    return (
        f"Titel: {metadata['title']}\n"
        f"Kanal: {metadata['channel']}\n"
        f"Hochgeladen am: {upload_date_formatted}\n"
        f"Länge: {duration_min} Minuten\n"
        f"Beschreibung (Auszug): {metadata['description'][:300]}..."
    )


# Test
print(get_video_metadata_tool.invoke({}))

Titel: Nutrients For Brain Health & Performance | Huberman Lab Podcast #42
Kanal: Andrew Huberman
Hochgeladen am: 2021-10-18
Länge: 101 Minuten
Beschreibung (Auszug): This episode I describe science-supported nutrients for brain and performance (cognition) and for nervous system health generally.

I describe 10 tools for this purpose, including specific amounts and sources for Omega-3 fatty acids which make up the "structural fat" of neurons (nerve cells) and all...


## Tags/Themen generieren (fehlender Teil)

Im Gegensatz zu den anderen Metadaten gibt YouTube keine Themen-Tags direkt heraus -- wir lassen das LLM sie aus dem Transcript-Inhalt ableiten.


In [19]:
def generate_topic_tags(full_text: str, n_tags: int = 5) -> list:
    prompt = f"""Based on this video transcript, generate {n_tags} short topic tags (1-3 words each) 
that describe the main subjects covered. Return ONLY the tags, one per line, no numbering.

Transcript excerpt:
{full_text[:3000]}"""  # nur die ersten 3000 Zeichen reichen, um das Thema zu erkennen

    response = llm.invoke(prompt)
    tags = [line.strip() for line in response.content.split("\n") if line.strip()]
    return tags


full_text = get_full_transcript_text()
tags = generate_topic_tags(full_text)

print("Generierte Tags:", tags)

# Zu den bestehenden Metadaten hinzufügen und erneut speichern
video_metadata["tags"] = tags
with open(f"../data/video_metadata/{final_video_id}.json", "w", encoding="utf-8") as f:
    json.dump(video_metadata, f, ensure_ascii=False, indent=2)

print("✅ Tags zur Metadata-Datei hinzugefügt")

Generierte Tags: ['Food and Brain', 'Cognition', 'Gut Signals', 'Metabolic Accessibility', 'Intermittent Fasting']
✅ Tags zur Metadata-Datei hinzugefügt


aktualisierte Version des Tools + (Tags/Topic Metdata)


In [20]:
@tool
def get_video_metadata_tool() -> str:
    """Gibt Informationen über das Video selbst zurück: Titel, Kanal, Upload-Datum, Länge, Themen-Tags, Beschreibung.
    Nutze dieses Tool, wenn der User nach dem Video selbst fragt (nicht nach seinem Inhalt) -- 
    z.B. 'Wie heißt das Video?', 'Wie lang ist es?', 'Von wem ist es?', 'Worum geht es grob?'."""

    with open(f"../data/video_metadata/{final_video_id}.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

    duration_min = metadata["duration_seconds"] // 60
    upload_date_formatted = f"{metadata['upload_date'][:4]}-{metadata['upload_date'][4:6]}-{metadata['upload_date'][6:]}"
    tags_str = ", ".join(metadata.get("tags", []))

    return (
        f"Titel: {metadata['title']}\n"
        f"Kanal: {metadata['channel']}\n"
        f"Hochgeladen am: {upload_date_formatted}\n"
        f"Länge: {duration_min} Minuten\n"
        f"Themen: {tags_str}\n"
        f"Beschreibung (Auszug): {metadata['description'][:300]}..."
    )


# Test
print(get_video_metadata_tool.invoke({}))

Titel: Nutrients For Brain Health & Performance | Huberman Lab Podcast #42
Kanal: Andrew Huberman
Hochgeladen am: 2021-10-18
Länge: 101 Minuten
Themen: Food and Brain, Cognition, Gut Signals, Metabolic Accessibility, Intermittent Fasting
Beschreibung (Auszug): This episode I describe science-supported nutrients for brain and performance (cognition) and for nervous system health generally.

I describe 10 tools for this purpose, including specific amounts and sources for Omega-3 fatty acids which make up the "structural fat" of neurons (nerve cells) and all...


## Fact-Check-Tool (Ansatz A: LLM-Wissen)

Holt eine relevante Aussage aus dem Video, lässt das LLM sie separat anhand seines trainierten Wissens bewerten, gibt eine Ampel-Einschätzung zurück.

**Wichtig, ehrlich benannt:** Kein Abgleich gegen eine verifizierte externe Datenbank

Das LLM nutzt sein eigenes, antrainiertes Wissen.

Bonus-Ausbau (externe Nutrition-KB) separat geplant.


In [21]:
@tool
def fact_check_tool(claim_or_topic: str) -> str:
    """Prüft eine Ernährungs-/Fitness-Behauptung aus dem Video auf wissenschaftliche Plausibilität.
    Nutze dieses Tool, wenn der User wissen will, ob etwas 'stimmt', 'wissenschaftlich belegt' ist, 
    oder wie vertrauenswürdig eine Aussage im Video ist -- z.B. 'Stimmt es, dass Kreatin die Kognition verbessert?'."""

    # Schritt 1: relevante Aussage aus dem Video holen (Retrieval, wie gehabt)
    video_chunks = search_video(claim_or_topic, n_results=2)
    video_context = "\n".join(c["text"] for c in video_chunks)

    # Schritt 2: LLM bittet, das UNABHÄNGIG vom Video zu bewerten
    prompt = f"""Du bist ein wissenschaftlicher Fact-Checker im Bereich Ernährung/Fitness.

Folgende Aussage stammt aus einem YouTube-Video:
"{video_context}"

Bewerte diese Aussage anhand deines wissenschaftlichen Wissens. Antworte in genau diesem Format:

BEWERTUNG: [Weitgehend bestätigt / Teilweise bestätigt / Umstritten / Nicht ausreichend belegt]
BEGRÜNDUNG: [2-3 Sätze, warum]
HINWEIS: Diese Einschätzung basiert auf allgemeinem KI-Wissen, nicht auf einer geprüften externen Datenbank."""

    response = llm.invoke(prompt)
    return response.content


# Test
result = fact_check_tool.invoke({"claim_or_topic": "creatine improves cognition"})
print(result)

BEWERTUNG: Weitgehend bestätigt  
BEGRÜNDUNG: Creatin ist ein natürlich vorkommendes Molekül, das in Fleisch und Fisch vorkommt und auch als Nahrungsergänzungsmittel erhältlich ist. Studien haben gezeigt, dass Creatin sowohl die körperliche Leistungsfähigkeit als auch die kognitive Funktion unterstützen kann, insbesondere bei Personen, die keine ausreichenden Mengen aus der Nahrung erhalten. Die Aussage über die Verwendung von Creatin als "Baseline-Insurance-Policy" ist subjektiv, aber die potenziellen Vorteile sind in der wissenschaftlichen Literatur gut dokumentiert.  
HINWEIS: Diese Einschätzung basiert auf allgemeinem KI-Wissen, nicht auf einer geprüften externen Datenbank.
